In [1]:
from lab3 import *

## Функция 2. Аппроксимация функции «Шестигорбый верблюд»

В качестве второй функции используется функция «Шестигорбый верблюд», зависящая от двух переменных:

```python
def camel_back_6(x, y):
    term1 = (4 - 2.1*x**2 + (x**4)/3)*x**2
    term2 = x*y
    term3 = (-4 + 4*y**2)*y**2
    return term1 + term2 + term3
```

Область определения: x1 ∈ [-3; 3], x2 ∈ [-2; 2]. Для аппроксимации использовались нейронные сети MLP, RBF и ANFIS. Для оценки качества моделей применялись MSE, RMSE, MAE, R², MAPE и процент точек, попавших в заданный допуск.

### Сравнение моделей

| Модель                                        | MSE | RMSE | MAE | R² | MAPE, % | Качество, % |
|-----------------------------------------------|---:|---:|---:|---:|---:|---:|
| sklearn MLP                                   | 0.8833 | 0.9398 | 0.3460 | 0.9914 | 33.18 | 94.67 |
| PyTorch MLP                                   | 2.3095 | 1.5197 | 0.3494 | 0.9776 | 24.48 | 95.00 |
| RBF (МНК)                                     | 18.0696 | 4.2508 | 0.8324 | 0.8245 | 32.88 | 88.67 |
| RBF Adam                                      | 68.9986 | 8.3065 | 2.5948 | 0.3299 | 63.51 | 64.83 |
| RBF Adam (стандартизация)                     | 22.3158 | 4.7240 | 1.1163 | 0.7833 | 44.29 | 78.83 |
| ANFIS                                         | 3.8180 | 1.9540 | 0.6199 | 0.9629 | 43.62 | 88.92 |
| RBF Adam (равномерное)                        | 532.0958 | 23.0672 | 11.8737 | 0.1536 | 83.73 | 37.83 |
| RBF Adam (стандартизация, равномерное) | 19.2576 | 4.3883 | 2.5942 | 0.9694 | 47.11 | 68.42 |

### Вывод

Наилучший результат показала модель MLPRegressor из библиотеки sklearn. Она имеет наибольший коэффициент детерминации R² = 0.9914 и наименьшие значения MSE = 0.8833, RMSE = 0.9398 и MAE = 0.3460. Это свидетельствует о высокой точности аппроксимации функции на тестовой выборке.

Модель PyTorch MLP показала близкий результат и немного превзошла sklearn по проценту попадания в заданный допуск (95% против 94.67%). Однако по основным показателям ошибки и R² лучшей является sklearn MLP.

Модели RBF показали менее стабильные результаты. Наиболее успешным оказался вариант с обучением методом наименьших квадратов. ANFIS также обеспечила достаточно высокое качество аппроксимации, но уступила MLP.

Таким образом, для аппроксимации функции «Шестигорбый верблюд» наиболее подходящей среди рассмотренных моделей является sklearn MLPRegressor.

In [2]:
print("\n--- Шестигорбый верблюд ---")


def camel_back_6(x, y):
    term1 = (4 - 2.1 * x ** 2 + (x ** 4) / 3) * x ** 2
    term2 = x * y
    term3 = (-4 + 4 * y ** 2) * y ** 2
    return term1 + term2 + term3


rng = np.random.default_rng(123)

def truncated_normal(size, low, high, rng):
    values = rng.normal(0, 1, size)
    mask = (values < low) | (values > high)

    while mask.any():
        values[mask] = rng.normal(0, 1, mask.sum())
        mask = (values < low) | (values > high)

    return values

X_f2 = np.column_stack([
    truncated_normal(3000, -3, 3, rng),
    truncated_normal(3000, -2, 2, rng)
])

x1_f2 = X_f2[:, 0]
x2_f2 = X_f2[:, 1]
y_f2 = camel_back_6(x1_f2, x2_f2)

X_train_f2, X_test_f2, y_train_f2, y_test_f2 = train_test_split(
    X_f2, y_f2, test_size=0.4, random_state=123
)

deviation = 0.1 * np.std(y_train_f2)


--- Шестигорбый верблюд ---


In [3]:
model_path = "best_net_f2.pkl"

if os.path.exists(model_path):
    best_net_f2 = joblib.load(model_path)
    print("Загружена сохранённая модель.")

else:
    net_f2 = MLPRegressor(random_state=0)

    params_f2 = {
        "hidden_layer_sizes": [(60, 60, 60)],
        "activation": ["relu", "tanh"],
        "solver": ["adam"],
        "learning_rate_init": [0.0001, 0.0005, 0.001],
        "alpha": [0.001],
        "max_iter": [1000]
    }

    random_search = RandomizedSearchCV(
        estimator=net_f2,
        param_distributions=params_f2,
        n_iter=20,
        scoring="r2",
        cv=5,
        random_state=42,
        n_jobs=-1
    )

    random_search.fit(X_train_f2, y_train_f2)

    best_net_f2 = random_search.best_estimator_

    joblib.dump(best_net_f2, model_path)

    print("Модель обучена и сохранена.")
    print("Лучшие параметры:", random_search.best_params_)
    print("Лучший R2 на кросс-валидации:", random_search.best_score_)



pred_f2 = best_net_f2.predict(X_test_f2)
print("R2 на тесте:", r2_score(y_test_f2, pred_f2))

m = MetricsCalculator()

metrics_f2 = m.calculate_comprehensive_metrics(
    y_true=y_test_f2,
    y_pred=pred_f2,
    model_name="sklearn: шестигорбый верблюд"
)

quality_f2 = m.approximation_quality(
    deviation=deviation,
    y_pred=pred_f2,
    y_true=y_test_f2
)

print(f"Качество аппроксимации: {quality_f2:.2f}%")


Загружена сохранённая модель.
R2 на тесте: 0.9914214832814287

sklearn: шестигорбый верблюд - МЕТРИКИ КАЧЕСТВА:
----------------------------------------
MSE:  0.883274
RMSE: 0.939827
MAE:  0.346025
R²:   0.991421
MAPE: 33.1837%
Качество аппроксимации: 94.67%


In [4]:
visualize_and_evaluate_3d_model(
    model=best_net_f2,
    X_test=X_test_f2,
    y_test=y_test_f2,
    y_pred=pred_f2,
    function=camel_back_6,
    title_name="Аппроксимация функции «Шестигорбый верблюд»"
)

In [5]:
inputs = torch.tensor(X_train_f2, dtype=torch.float32)
targets = torch.tensor(
    np.asarray(y_train_f2).reshape(-1),
    dtype=torch.float32
)

model_path = "best_MLPnet_f2.pth"

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size[0])
        self.fc2 = nn.Linear(hidden_size[0], hidden_size[1])
        self.fc3 = nn.Linear(hidden_size[1], hidden_size[2])
        self.fc4 = nn.Linear(hidden_size[2], output_size)

    def forward(self, x):
        out = F.relu(self.fc1(x))
        out = F.relu(self.fc2(out))
        out = F.relu(self.fc3(out))
        out = self.fc4(out)
        return out


input_size = inputs.size()[1]
hidden_size = (60,60,60)
output_size = 1
num_epoch = 1000
learning_rate = 0.0001

model = MLP(input_size=input_size, hidden_size=hidden_size,
            output_size=output_size)


if os.path.exists(model_path):
    model.load_state_dict(
        torch.load(model_path, weights_only=True)
    )
    print("Загружена сохранённая модель.")
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    loss_fct = nn.MSELoss()

    training_samples = utils_data.TensorDataset(inputs, targets)
    data_loader_trn = utils_data.DataLoader(training_samples, batch_size=150, drop_last=False, shuffle=True)

    for epoch in range(num_epoch):
        c_loss = 0
        for batch_idx, (data, target) in enumerate(data_loader_trn):
            tr_x, tr_y = data.float(), target.float()

            pred = model(tr_x)
            loss = loss_fct(pred, tr_y.unsqueeze(1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            c_loss += loss.item()

        if epoch % 10 == 0:
            print('Epoch [%d/%d], Loss: %.4f'
                  % (epoch + 1, num_epoch, c_loss))

    torch.save(model.state_dict(), model_path)
    print("Модель обучена и сохранена.")

model.eval()

with torch.no_grad():
    test_inputs = torch.tensor(X_test_f2, dtype=torch.float32)
    final_prediction = model(test_inputs)

final_pred_np = final_prediction.cpu().numpy().ravel()
targets_np = np.asarray(y_test_f2).ravel()

print("Корреляция:", np.corrcoef(final_pred_np, targets_np)[0, 1])

Загружена сохранённая модель.
Корреляция: 0.9895488904551866


In [6]:
metrics_f2 = m.calculate_comprehensive_metrics(
    y_true=targets_np,
    y_pred=final_pred_np,
    model_name="Аппроксимация функции «Шестигорбый верблюд»"
)

quality_f2 = m.approximation_quality(
    deviation=deviation,
    y_pred=final_pred_np,
    y_true=targets_np
)

print(f"Качество аппроксимации: {quality_f2:.2f}%")


Аппроксимация функции «Шестигорбый верблюд» - МЕТРИКИ КАЧЕСТВА:
----------------------------------------
MSE:  2.309528
RMSE: 1.519713
MAE:  0.349358
R²:   0.977569
MAPE: 24.4829%
Качество аппроксимации: 95.00%


In [7]:
visualize_and_evaluate_3d_model(
    model=model,
    X_test=X_test_f2,
    y_test=y_test_f2,
    y_pred=final_pred_np,
    function=camel_back_6,
    title_name="Аппроксимация функции «Шестигорбый верблюд»"
)

In [8]:
model_path = "best_RBF_f2.pkl"

if os.path.exists(model_path):
    rbf_f2 = joblib.load(model_path)
    print("Загружена сохранённая модель RBF.")

else:

    rbf_f2 = RBF(
        indim=2,
        numCenters=170,
        outdim=1
    )

    rbf_f2.train(
        X_train_f2,
        np.asarray(y_train_f2).reshape(-1, 1),
        epochs=1000, lr=0.0001,
    )

    joblib.dump(rbf_f2, model_path)
    print("Модель RBF обучена и сохранена.")



for i in range(rbf_f2.numCenters):
    print(
        f"Нейрон {i + 1}: "
        f"центр = {rbf_f2.centers[i]}, "
        f"sigma = {rbf_f2.sigmas[i]:.4f}, "
        f"beta = {1 / (2 * rbf_f2.sigmas[i] ** 2):.4f}"
    )

pred_test_rbf_f2 = rbf_f2.test(X_test_f2).ravel()

print(
    "R2 на тесте:",
    r2_score(y_test_f2, pred_test_rbf_f2)
)

m = MetricsCalculator()

metrics_rbf_f2 = m.calculate_comprehensive_metrics(
    y_true=y_test_f2,
    y_pred=pred_test_rbf_f2,
    model_name="RBF: шестигорбый верблюд"
)

quality_rbf_f2 = m.approximation_quality(
    deviation=deviation,
    y_pred=pred_test_rbf_f2,
    y_true=y_test_f2
)

print(f"Качество аппроксимации RBF: {quality_rbf_f2:.2f}%")

Загружена сохранённая модель RBF.
Нейрон 1: центр = [0.46309986 0.53027162], sigma = 0.2284, beta = 9.5805
Нейрон 2: центр = [-0.52297493 -0.36547796], sigma = 0.2633, beta = 7.2147
Нейрон 3: центр = [ 0.97413676 -0.65766433], sigma = 0.2835, beta = 6.2201
Нейрон 4: центр = [-1.77657831  1.16167691], sigma = 0.3780, beta = 3.4998
Нейрон 5: центр = [-0.10165346  1.56831035], sigma = 0.3502, beta = 4.0765
Нейрон 6: центр = [1.49727842 0.7400486 ], sigma = 0.3246, beta = 4.7450
Нейрон 7: центр = [-0.46105295 -1.49690707], sigma = 0.3198, beta = 4.8884
Нейрон 8: центр = [-0.94149553  0.79497107], sigma = 0.3094, beta = 5.2225
Нейрон 9: центр = [-1.61937475  0.01954837], sigma = 0.2908, beta = 5.9145
Нейрон 10: центр = [-1.1881316 -0.939114 ], sigma = 0.3233, beta = 4.7839
Нейрон 11: центр = [ 0.06418947 -0.16054805], sigma = 0.2291, beta = 9.5271
Нейрон 12: центр = [ 0.81839565 -1.73395179], sigma = 0.3996, beta = 3.1320
Нейрон 13: центр = [ 1.5666876 -0.5663298], sigma = 0.3286, beta = 4.

In [9]:
visualize_and_evaluate_3d_model(
    model=rbf_f2,
    X_test=X_test_f2,
    y_test=y_test_f2,
    y_pred=pred_test_rbf_f2,
    function=camel_back_6,
    title_name="RBF: шестигорбый верблюд"
)

In [10]:
model_path = "best_RBFAdam_f2.pkl"

if os.path.exists(model_path):
    rbf_f2 = joblib.load(model_path)
    print("Загружена сохранённая модель RBF.")

else:

    rbf_f2 = RBF(
        indim=2,
        numCenters=170,
        outdim=1
    )

    rbf_f2.train(
        X_train_f2,
        np.asarray(y_train_f2).reshape(-1, 1),
        epochs=2000, lr=0.0001,
        adam=True
    )
    joblib.dump(rbf_f2, model_path)
    print("Модель RBF обучена и сохранена.")


for i in range(rbf_f2.numCenters):
    print(
        f"Нейрон {i + 1}: "
        f"центр = {rbf_f2.centers[i]}, "
        f"sigma = {rbf_f2.sigmas[i]:.4f}, "
        f"beta = {1 / (2 * rbf_f2.sigmas[i] ** 2):.4f}"
    )


pred_test_rbf_f2 = rbf_f2.test(X_test_f2).ravel()

print(
    "R2 на тесте:",
    r2_score(y_test_f2, pred_test_rbf_f2)
)

m = MetricsCalculator()

metrics_rbf_f2 = m.calculate_comprehensive_metrics(
    y_true=y_test_f2,
    y_pred=pred_test_rbf_f2,
    model_name="RBF: шестигорбый верблюд"
)

quality_rbf_f2 = m.approximation_quality(
    deviation=deviation,
    y_pred=pred_test_rbf_f2,
    y_true=y_test_f2
)

print(f"Качество аппроксимации RBF: {quality_rbf_f2:.2f}%")

Загружена сохранённая модель RBF.
Нейрон 1: центр = [0.46309986 0.53027162], sigma = 0.2284, beta = 9.5805
Нейрон 2: центр = [-0.52297493 -0.36547796], sigma = 0.2633, beta = 7.2147
Нейрон 3: центр = [ 0.97413676 -0.65766433], sigma = 0.2835, beta = 6.2201
Нейрон 4: центр = [-1.77657831  1.16167691], sigma = 0.3780, beta = 3.4998
Нейрон 5: центр = [-0.10165346  1.56831035], sigma = 0.3502, beta = 4.0765
Нейрон 6: центр = [1.49727842 0.7400486 ], sigma = 0.3246, beta = 4.7450
Нейрон 7: центр = [-0.46105295 -1.49690707], sigma = 0.3198, beta = 4.8884
Нейрон 8: центр = [-0.94149553  0.79497107], sigma = 0.3094, beta = 5.2225
Нейрон 9: центр = [-1.61937475  0.01954837], sigma = 0.2908, beta = 5.9145
Нейрон 10: центр = [-1.1881316 -0.939114 ], sigma = 0.3233, beta = 4.7839
Нейрон 11: центр = [ 0.06418947 -0.16054805], sigma = 0.2291, beta = 9.5271
Нейрон 12: центр = [ 0.81839565 -1.73395179], sigma = 0.3996, beta = 3.1320
Нейрон 13: центр = [ 1.5666876 -0.5663298], sigma = 0.3286, beta = 4.

In [11]:
visualize_and_evaluate_3d_model(
    model=rbf_f2,
    X_test=X_test_f2,
    y_test=y_test_f2,
    y_pred=pred_test_rbf_f2,
    function=camel_back_6,
    title_name="RBF: шестигорбый верблюд"
)

In [12]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train_f2)
X_test_scaled = scaler_X.transform(X_test_f2)

y_train_scaled = scaler_y.fit_transform(
    np.asarray(y_train_f2).reshape(-1, 1)
)

model_path = "best_RBFAdam_f2_scaled.pkl"

if os.path.exists(model_path):
    checkpoint = joblib.load(model_path)
    rbf_f2 = checkpoint["model"]
    scaler_X = checkpoint["scaler_X"]
    scaler_y = checkpoint["scaler_y"]

    print("Загружена сохранённая модель RBF.")

else:
    rbf_f2 = RBF(
        indim=2,
        numCenters=170,
        outdim=1
    )

    rbf_f2.train(
        X_train_scaled,
        y_train_scaled,
        epochs=1000,
        lr=0.001,
        adam=True
    )

    joblib.dump({
        "model": rbf_f2,
        "scaler_X": scaler_X,
        "scaler_y": scaler_y
    }, model_path)

    print("Модель RBF обучена и сохранена.")


pred_test_scaled = rbf_f2.test(X_test_scaled)

pred_test_rbf_f2 = scaler_y.inverse_transform(
    pred_test_scaled
).ravel()

print(
    "R2 на тесте:",
    r2_score(y_test_f2, pred_test_rbf_f2)
)

m = MetricsCalculator()

metrics_rbf_f2 = m.calculate_comprehensive_metrics(
    y_true=y_test_f2,
    y_pred=pred_test_rbf_f2,
    model_name="RBF Adam: стандартизированные данные"
)

quality_rbf_f2 = m.approximation_quality(
    deviation=deviation,
    y_pred=pred_test_rbf_f2,
    y_true=y_test_f2
)

print(f"Качество аппроксимации RBF: {quality_rbf_f2:.2f}%")

Загружена сохранённая модель RBF.
R2 на тесте: 0.783264788753613

RBF Adam: стандартизированные данные - МЕТРИКИ КАЧЕСТВА:
----------------------------------------
MSE:  22.315812
RMSE: 4.723961
MAE:  1.116332
R²:   0.783265
MAPE: 44.2925%
Качество аппроксимации RBF: 78.83%


In [13]:
visualize_and_evaluate_3d_model(
    model=rbf_f2,
    X_test=X_test_f2,
    y_test=y_test_f2,
    y_pred=pred_test_rbf_f2,
    function=camel_back_6,
    title_name="RBF: шестигорбый верблюд",
    scaler_X=scaler_X,
    scaler_y=scaler_y
)

In [14]:
X_anfis_train, X_anfis_valid, y_anfis_train, y_anfis_valid = train_test_split(
    X_train_f2,
    y_train_f2,
    test_size=0.15,
    random_state=123
)

X_anfis_train = torch.as_tensor(X_anfis_train, dtype=torch.float32)
X_anfis_valid = torch.as_tensor(X_anfis_valid, dtype=torch.float32)

y_anfis_train = torch.as_tensor(
    np.asarray(y_anfis_train).reshape(-1, 1),
    dtype=torch.float32
)

y_anfis_valid = torch.as_tensor(
    np.asarray(y_anfis_valid).reshape(-1, 1),
    dtype=torch.float32
)

model_path = "best_ANFIS_f2.pth"

membfuncs = [
    {
        'function': 'gaussian',
        'n_memb': 4,
        'params': {
            'mu': {
                'value': [-1.5, -0.5, 0.5, 1.5],
                'trainable': True
            },
            'sigma': {
                'value': [0.8, 0.8, 0.8, 0.8],
                'trainable': True
            }
        }
    },
    {
        'function': 'gaussian',
        'n_memb': 4,
        'params': {
            'mu': {
                'value': [-1.5, -0.5, 0.5, 1.5],
                'trainable': True
            },
            'sigma': {
                'value': [0.8, 0.8, 0.8, 0.8],
                'trainable': True
            }
        }
    }
]


fis = SANFIS(
    membfuncs=membfuncs,
    n_input=2,
    scale='Std'
)

print_rules(fis, 'Правила до обучения')


if os.path.exists(model_path):
    checkpoint = torch.load(
        model_path,
        map_location='cpu',
        weights_only=False
    )

    fis.load_state_dict(checkpoint['model_state_dict'])

    fis.scaler = checkpoint['scaler']

else:

    loss_function = torch.nn.MSELoss(reduction='mean')
    optimizer = torch.optim.Adam(fis.parameters(), lr=0.001)

    history = fis.fit(
        train_data=[X_anfis_train, y_anfis_train],
        valid_data=[X_anfis_valid, y_anfis_valid],
        optimizer=optimizer,
        loss_function=loss_function,
        epochs=1000
    )
    torch.save({
        'model_state_dict': fis.state_dict(),
        'scaler': fis.scaler,
        'scaling_params': fis.scaling_params
    }, model_path)

    print("Загружена сохранённая модель ANFIS.")

print_rules(fis, 'Правила после обучения')

X_test_tensor = torch.as_tensor(
    X_test_f2,
    dtype=torch.float32
)

y_pred_anfis_f2 = np.asarray(
    fis.predict(X_test_tensor)
).reshape(-1)



Правила до обучения
Правило 1: ЕСЛИ x принадлежит A1 (mu=-1.5000, sigma=0.8000), ТО y = -0.4708x + 0.0952
Правило 2: ЕСЛИ x принадлежит A2 (mu=-0.5000, sigma=0.8000), ТО y = -0.0294x + 0.3855
Правило 3: ЕСЛИ x принадлежит A3 (mu=0.5000, sigma=0.8000), ТО y = -0.0406x - 0.2438
Правило 4: ЕСЛИ x принадлежит A4 (mu=1.5000, sigma=0.8000), ТО y = 0.4266x + 0.2195

Правила после обучения
Правило 1: ЕСЛИ x принадлежит A1 (mu=-3.8534, sigma=0.9862), ТО y = -0.6379x + 1.9217
Правило 2: ЕСЛИ x принадлежит A2 (mu=-1.0235, sigma=1.0780), ТО y = -1.5644x + 0.4863
Правило 3: ЕСЛИ x принадлежит A3 (mu=-0.3056, sigma=0.9160), ТО y = -2.2494x + 1.0106
Правило 4: ЕСЛИ x принадлежит A4 (mu=1.2931, sigma=0.6341), ТО y = -0.7186x + 1.0386


In [15]:
m = MetricsCalculator()

metrics_anfis_f2 = m.calculate_comprehensive_metrics(
    y_true=y_test_f2,
    y_pred=y_pred_anfis_f2,
    model_name="ANFIS: шестигорбый верблюд"
)

quality_anfis_f2 = m.approximation_quality(
    deviation=deviation,
    y_true=y_test_f2,
    y_pred=y_pred_anfis_f2
)

print(f"Качество  ANFIS: {quality_anfis_f2:.2f}%")


ANFIS: шестигорбый верблюд - МЕТРИКИ КАЧЕСТВА:
----------------------------------------
MSE:  3.817979
RMSE: 1.953965
MAE:  0.619879
R²:   0.962919
MAPE: 43.6233%
Качество  ANFIS: 88.92%


In [16]:
visualize_and_evaluate_3d_model(
    model=fis,
    X_test=X_test_f2,
    y_test=y_test_f2,
    y_pred=y_pred_anfis_f2,
    function=camel_back_6,
    title_name="ANFIS: шестигорбый верблюд"
)

In [17]:
X_f2 = np.column_stack([
    rng.uniform(-3, 3, 3000),
    rng.uniform(-2, 2, 3000),
])

x1_f2 = X_f2[:, 0]
x2_f2 = X_f2[:, 1]
y_f2 = camel_back_6(x1_f2, x2_f2)

X_train_f2, X_test_f2, y_train_f2, y_test_f2 = train_test_split(
    X_f2, y_f2, test_size=0.4, random_state=123
)

deviation = 0.1 * np.std(y_train_f2)

In [18]:
model_path = "best_RBFAdam_uniform_f2.pkl"

if os.path.exists(model_path):
    rbf_f2 = joblib.load(model_path)
    print("Загружена сохранённая модель RBF.")

else:

    rbf_f2 = RBF(
        indim=2,
        numCenters=170,
        outdim=1
    )

    rbf_f2.train(
        X_train_f2,
        np.asarray(y_train_f2).reshape(-1, 1),
        epochs=2000, lr=0.0001,
        adam=True
    )
    joblib.dump(rbf_f2, model_path)
    print("Модель RBF обучена и сохранена.")

for i in range(rbf_f2.numCenters):
    print(
        f"Нейрон {i + 1}: "
        f"центр = {rbf_f2.centers[i]}, "
        f"sigma = {rbf_f2.sigmas[i]:.4f}, "
        f"beta = {1 / (2 * rbf_f2.sigmas[i] ** 2):.4f}"
    )

pred_test_rbf_f2 = rbf_f2.test(X_test_f2).ravel()

print(
    "R2 на тесте:",
    r2_score(y_test_f2, pred_test_rbf_f2)
)

m = MetricsCalculator()

metrics_rbf_f2 = m.calculate_comprehensive_metrics(
    y_true=y_test_f2,
    y_pred=pred_test_rbf_f2,
    model_name="RBF: шестигорбый верблюд"
)

quality_rbf_f2 = m.approximation_quality(
    deviation=deviation,
    y_pred=pred_test_rbf_f2,
    y_true=y_test_f2
)

print(f"Качество аппроксимации RBF: {quality_rbf_f2:.2f}%")
visualize_and_evaluate_3d_model(
    model=rbf_f2,
    X_test=X_test_f2,
    y_test=y_test_f2,
    y_pred=pred_test_rbf_f2,
    function=camel_back_6,
    title_name="RBF: шестигорбый верблюд"
)

Загружена сохранённая модель RBF.
Нейрон 1: центр = [-2.76381643 -1.85265654], sigma = 0.4721, beta = 2.2434
Нейрон 2: центр = [1.51384739 1.14076384], sigma = 0.3231, beta = 4.7891
Нейрон 3: центр = [-1.38896305  0.30160284], sigma = 0.3577, beta = 3.9071
Нейрон 4: центр = [ 1.27733999 -0.9301047 ], sigma = 0.3651, beta = 3.7517
Нейрон 5: центр = [-0.70364196 -0.86113539], sigma = 0.3766, beta = 3.5257
Нейрон 6: центр = [-1.12320994  1.76843864], sigma = 0.3579, beta = 3.9026
Нейрон 7: центр = [ 2.51613919 -1.04450948], sigma = 0.3424, beta = 4.2650
Нейрон 8: центр = [-2.28048604  0.95147799], sigma = 0.3553, beta = 3.9603
Нейрон 9: центр = [0.14160059 0.02584404], sigma = 0.3976, beta = 3.1626
Нейрон 10: центр = [2.42293095 1.86736424], sigma = 0.4921, beta = 2.0651
Нейрон 11: центр = [2.58465651 0.32679098], sigma = 0.3391, beta = 4.3481
Нейрон 12: центр = [0.20624351 1.36664521], sigma = 0.3322, beta = 4.5311
Нейрон 13: центр = [-2.92120714 -0.18661687], sigma = 0.3205, beta = 4.86

In [19]:

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train_f2)
X_test_scaled = scaler_X.transform(X_test_f2)

y_train_scaled = scaler_y.fit_transform(
    np.asarray(y_train_f2).reshape(-1, 1)
)

model_path = "best_RBFAdam_uniform_f2_scaled.pkl"

if os.path.exists(model_path):
    checkpoint = joblib.load(model_path)
    rbf_f2 = checkpoint["model"]
    scaler_X = checkpoint["scaler_X"]
    scaler_y = checkpoint["scaler_y"]

    print("Загружена сохранённая модель RBF.")

else:
    rbf_f2 = RBF(
        indim=2,
        numCenters=170,
        outdim=1
    )

    rbf_f2.train(
        X_train_scaled,
        y_train_scaled,
        epochs=2000,
        lr=0.001,
        adam=True
    )

    joblib.dump({
        "model": rbf_f2,
        "scaler_X": scaler_X,
        "scaler_y": scaler_y
    }, model_path)

    print("Модель RBF обучена и сохранена.")

pred_test_scaled = rbf_f2.test(X_test_scaled)

pred_test_rbf_f2 = scaler_y.inverse_transform(
    pred_test_scaled
).ravel()

print(
    "R2 на тесте:",
    r2_score(y_test_f2, pred_test_rbf_f2)
)

m = MetricsCalculator()

metrics_rbf_f2 = m.calculate_comprehensive_metrics(
    y_true=y_test_f2,
    y_pred=pred_test_rbf_f2,
    model_name="RBF Adam: стандартизированные данные"
)

quality_rbf_f2 = m.approximation_quality(
    deviation=deviation,
    y_pred=pred_test_rbf_f2,
    y_true=y_test_f2
)

print(f"Качество аппроксимации RBF: {quality_rbf_f2:.2f}%")


Загружена сохранённая модель RBF.
R2 на тесте: 0.9693681451377437

RBF Adam: стандартизированные данные - МЕТРИКИ КАЧЕСТВА:
----------------------------------------
MSE:  19.257580
RMSE: 4.388346
MAE:  2.594236
R²:   0.969368
MAPE: 47.1111%
Качество аппроксимации RBF: 68.42%


In [20]:
visualize_and_evaluate_3d_model(
    model=rbf_f2,
    X_test=X_test_f2,
    y_test=y_test_f2,
    y_pred=pred_test_rbf_f2,
    function=camel_back_6,
    title_name="RBF: шестигорбый верблюд",
    scaler_X=scaler_X,
    scaler_y=scaler_y
)